$$E_R^{local}=E_{baseR}+S_{noise}\cdot N\cdot \left(1+k\cdot N\right)$$

$E_{baseR}$取1200

120dB的等值线半径对于超重型火箭约为8-10km，$S=3.14\times{10}^2=314km^2$，取$S_{noise}$=300

N为发射次数，计算的是一个基地一年的发射总量

1200为火箭基地本身存在对周遭生态环境的影响范围

k取	0.001 ~ 0.005

---

$$E_H^{local}=E_{Barrier}+E_{baseH}$$

$E_{baseH}$取2500，太空电梯港本身对周遭环境的影响，考虑半径30km得出2500


$$E_{Barrier}=\beta\cdot\delta\cdot\left(P\right)^\gamma$$

$\beta = 1000$

$\delta$，迁徙系数 0.43

$P$ 运行影响因子, 50

$\gamma$,取1.5


---

$$E_{O_3}=V_{gas}\cdot\rho No_x\cdot\Phi$$

火箭每次发射的污染情况

$V_{gas}$，高温区域体积 10^8 $m^3$

$\rho NO_x$，NOx产出率，也称泽尔多维奇生成系数 $0.01 kg/m^3$

$\Phi$，催化因子，少量NO可以持续破坏臭氧层 1000

---

$$E_{CO_2}=\epsilon\cdot(Monce+McycleLlife+f·Mmaintenance)$$

火箭每次发射的污染情况

$\epsilon = 40kgCO2/kg$

$M_{once}$，一次性组件质量 50t

$M_{cycle}$，复用组件质量 300t

$L_{life}$，复用寿命 300

$M_{maintenance}$，维护组件质量 300t

$f$，单次维护频率 0.05


优化目标

$$E_{total} = E_R^{local} + E_H^{local} + E_{O_3} + E_{CO_2}$$

使得最小

参考根据火箭和电梯的货物运输配比alpha，0和1，分别计算出每个子影响的基准值，然后再通过网格搜索法，查找出不同alpha配比下的方案的各个子影响的数值，并根据之前的基准值进行归一化后求和算出总的影响值，然后找到最小的影响方案。

## 代码实现

下面把你在上面写的四个子影响项都实现成函数，然后：

- 设定运输配比 $\alpha\ in[0,1]$ 其中 $\alpha$ 越大表示越多由电梯承担，火箭承担 $1-\alpha$
- 对每个 $\alpha\$ 计算：$E_R^{local}, E_H^{local}, E_{O_3}, E_{CO_2}$
- 用 $\alpha = 0$  与 $\alpha = 1$ 作为端点基准做 min-max 归一化
- 归一化后求和得到 $E_{total}^{norm}$，网格搜索找到最优 $\alpha = 0$

**说明：**

- 若某系统不启用(例如 $\alpha = 0 $ )不建电梯，或火箭年度发射次数 N=0 不建火箭基地）

你如果希望“只要建了就有基地影响（即使不用）”，把代码里的开关改掉即可。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 配置matplotlib
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3


In [ ]:
# ==================== 1) 可调参数 ====================

"""
完整参数配置
"""

# ==================== 全局目标 ====================
target_payload = 1e8 * 1000  # 1亿吨 (kg)
max_years = 300              # 最大仿真年数
ref_cost = 5e12              # 参考成本 $5T
ref_time = 40                # 参考时间 40年
alpha_weight = 0.5           # 成本与时间权衡系数

# ==================== 太空电梯 (SE) 参数 ====================
se_cap_design = 179000 * 3 * 1000  # 3个港口设计年运力 (kg)

# ==================== 十大火箭发射场数据 ====================
sites_data = [
    {'name': 'India (Satish Dhawan)', 'pl': 145000, 'vc': 300*145000, 'fc': 150e6, 'L': 4},
    {'name': 'China (Taiyuan)', 'pl': 125000, 'vc': 320*125000, 'fc': 200e6, 'L': 5},
    {'name': 'USA (SpaceX Texas)', 'pl': 145000, 'vc': 320*145000, 'fc': 350e6, 'L': 6},
    {'name': 'USA (Cape Canaveral)', 'pl': 140000, 'vc': 350*140000, 'fc': 400e6, 'L': 8},
    {'name': 'Kazakhstan (Baikonur)', 'pl': 120000, 'vc': 380*120000, 'fc': 250e6, 'L': 4},
    {'name': 'New Zealand (Mahia)', 'pl': 125000, 'vc': 400*125000, 'fc': 100e6, 'L': 3},
    {'name': 'Fr. Guiana (Kourou)', 'pl': 150000, 'vc': 450*150000, 'fc': 300e6, 'L': 3},
    {'name': 'USA (Vandenberg)', 'pl': 135000, 'vc': 450*135000, 'fc': 300e6, 'L': 4},
    {'name': 'USA (Wallops)', 'pl': 130000, 'vc': 480*130000, 'fc': 150e6, 'L': 2},
    {'name': 'USA (Kodiak)', 'pl': 100000, 'vc': 500*100000, 'fc': 100e6, 'L': 2},
]

num_sites = len(sites_data)
site_limits_annual = np.array([s['L'] * 365 for s in sites_data])  # 年度发射上限



# ==================== 显示配置摘要 ====================
print("=" * 100)
print("参数配置完成".center(100))
print("=" * 100)

print(f"\n全局目标:")
print(f"  总运输目标: {target_payload/1e9:.1f} 万吨")
print(f"  最大时间: {max_years} 年")

print(f"\n太空电梯:")
print(f"  设计年运力: {se_cap_design/1e9:.2f} 万吨")

print(f"\n火箭系统 (10个发射场):")
total_capacity = sum(s['L'] * 365 * s['pl'] for s in sites_data)
print(f"  总年运力: {total_capacity/1e9:.2f} 万吨")

print(f"\n发射场列表:")
print(f"{'序号':<4} {'发射场':<30} {'载荷(t)':<12} {'年运力(万t)':<15} {'固定成本($M)':<15}")
print("-" * 100)
for i, site in enumerate(sites_data):
    annual_cap = site['L'] * 365 * site['pl'] / 1e7
    print(f"{i:<4} {site['name']:<30} {site['pl']/1000:<12.0f} {annual_cap:<15.2f} {site['fc']/1e6:<15.0f}")

# ==================== E_R^{local}: 火箭基地本地噪声/生态影响 ====================
E_baseR = 1200
S_noise = 300  # km^2
k = 0.003      # 建议范围 0.001~0.005

# ==================== E_H^{local}: 电梯港本地影响 + 迁徙阻隔 ====================
E_baseH = 2500
beta = 1000
delta = 0.43
P_full = 50
gamma = 1.5

# ==================== E_{O3}: NOx 触发臭氧破坏（按“每次发射”累计）====================
V_gas = 1e8        # m^3
rho_NOx = 0.01     # kg/m^3
Phi = 1000

# ==================== E_{CO2}: 全生命周期 CO2（按“每次发射”累计）====================
# 注意：这里按你的注释理解为“把复用件在寿命内均摊到每次发射”
# 单位统一为 kg
epsilon = 40  # kgCO2/kg
M_once = 50_000
M_cycle = 300_000
L_life = 300
M_maintenance = 300_000
f_maint = 0.05

# ==================== 工程化开关（决定端点是否有基地本底影响）====================
# 若系统不启用，则不建基地/港：其本底影响记为0
ASSUME_NO_BASE_IF_UNUSED = True



In [ ]:
# ==================== 2) 影响函数（把α -> N -> 各子影响算出来）====================

def rocket_launches_per_year(alpha: float) -> int:
    """给定α，计算火箭系统在一年内需要的发射次数 N"""
    rocket_target = target_payload * (1-alpha)
    annual_num = 0
    years_num = 0 
    if alpha == 1.0:
        # 纯电梯方案：火箭基地全部作为应急备份（平时不启用）
        annual_num = 0
        years_num = target_payload/se_cap_design
    
    elif alpha == 0.0:
        # 纯火箭方案：全部基地满负荷运行
        for site_idx in range(num_sites):
            daily_launches = min(3, sites_data[site_idx]['L'])  # 每天最多3次或基地上限
            annual_num += daily_launches * 365  # 转换为年度次数
            
        years_num = target_payload/sum([min(3, s['L']) * 365 * s['pl'] for s in sites_data])
    
    else:
        # 混合方案：根据(1-α)计算需要的主力基地数，其余作为备份
        # 假设：每个基地每天3次，年运力 = 3 * 365 * 基地载荷
        total_rocket_annual_capacity = sum([min(3, s['L']) * 365 * s['pl'] for s in sites_data])
        required_rocket_payload = rocket_target  # 火箭需要承担的总量

        perfect_years_needed = rocket_target/total_rocket_annual_capacity

        years_SE = target_payload * alpha /se_cap_design
        
        # 计算需要几个基地（按固定成本从低到高选择）
        sorted_indices = np.argsort([s['fc'] for s in sites_data])
        cumulative_capacity = 0
        num_primary_sites = 0
        
        for idx in sorted_indices:
            daily_launches = min(3, sites_data[idx]['L'])  # 每天最多3次或基地上限
            site_annual_capacity = daily_launches * 365 * sites_data[idx]['pl']
            cumulative_capacity += site_annual_capacity
            num_primary_sites += 1
            
            # 计算需要多少年完成任务, 考虑备份运行火箭基地晚10年完成
            years_needed = required_rocket_payload / cumulative_capacity
            if years_needed <= perfect_years_needed + 20:
                break
        
        years_num = max(years_SE,years_needed)

        # 配置主力基地
        primary_sites = sorted_indices[:num_primary_sites]
        backup_sites = sorted_indices[num_primary_sites:]
        
        for site_idx in primary_sites:
            daily_launches = min(3, sites_data[site_idx]['L'])  # 每天最多3次或基地上限
            annual_num += daily_launches * 365  # 转换为年度发射次数！
    return annual_num, years_num


def E_R_local(N: int) -> float:
    """火箭基地本地影响（含噪声等值线覆盖 + 次数的非线性累积）。"""
    # 若 N=0 但仍认为基地存在，则只剩E_baseR
    return float(E_baseR + S_noise * N * (1.0 + k * N))


def E_barrier(alpha: float) -> float:
    """迁徙阻隔（假设随电梯承担比例α变化：P=P_full*α）。"""
    alpha = float(np.clip(alpha, 0.0, 1.0))
    P = P_full * (alpha+1)                                    # 电梯运输时的影响
    return float(beta * delta * (P ** gamma))


def E_H_local(alpha: float) -> float:
    """电梯港本地影响（港口本底 + 迁徙阻隔）。"""
    alpha = float(np.clip(alpha, 0.0, 1.0))
    return float(E_baseH + E_barrier(alpha))


def E_O3_total(N: int) -> float:
    """臭氧影响：按每次发射累积。"""
    if N <= 0:
        return 0.0
    per_launch = 5600 #V_gas * rho_NOx * Phi
    return float(N * per_launch)


def E_CO2_total(N: int) -> float:
    """CO2影响：按每次发射累积（含一次性+复用均摊+维护均摊）。"""
    if N <= 0:
        return 0.0
    mass_per_launch = M_once + (M_cycle / L_life) + (f_maint * M_maintenance)
    per_launch = epsilon * mass_per_launch
    return float(N * per_launch)


def impacts_by_alpha(alpha: float) -> dict:
    N , years= rocket_launches_per_year(alpha)
    return {
        'alpha': float(alpha),
        'N_launches': int(N),
        'Years_needed': int(years),
        'E_R_local': E_R_local(N),
        'E_H_local': E_H_local(alpha),
        'E_O3': E_O3_total(N),
        'E_CO2': E_CO2_total(N),
    }


def minmax_norm(x: float, x0: float, x1: float, eps: float = 1e-12) -> float:
    lo, hi = (x0, x1) if x0 <= x1 else (x1, x0)
    denom = hi - lo
    if denom < eps:
        return 0.0
    return float((x - lo) / denom)

print('函数定义完成')


In [ ]:
# ==================== 3) 端点基准 + α网格搜索（对标Q2“阶段一”）====================

alpha_values = np.linspace(0.0, 1.0, 21)  # 步长0.05

# 端点基准（α=0 与 α=1）
b0 = impacts_by_alpha(0.0)
b1 = impacts_by_alpha(1.0)

baseline = {
    'E_R_local': (b0['E_R_local'], b1['E_R_local']),
    'E_H_local': (b0['E_H_local'], b1['E_H_local']),
    'E_O3': (b0['E_O3'], b1['E_O3']),
    'E_CO2': (b0['E_CO2'], b1['E_CO2']),
}

rows = []
for a in alpha_values:
    imp = impacts_by_alpha(a)

    # imp['E_R_norm'] = minmax_norm(imp['E_R_local'], *baseline['E_R_local'])
    # imp['E_H_norm'] = minmax_norm(imp['E_H_local'], *baseline['E_H_local'])
    # imp['E_O3_norm'] = minmax_norm(imp['E_O3'], *baseline['E_O3'])
    # imp['E_CO2_norm'] = minmax_norm(imp['E_CO2'], *baseline['E_CO2'])

    # 求和
    imp['E_total'] = 1000000*imp['E_R_local'] + 1000000*imp['E_H_local'] + 5000*imp['E_O3'] + 0.2*imp['E_CO2']

    rows.append(imp)

df = pd.DataFrame(rows)

best_idx = df['E_total'].idxmin()
best = df.loc[best_idx].to_dict()

print('端点基准（α=0 / α=1）:')
print(pd.DataFrame({
    'alpha=0': b0,
    'alpha=1': b1,
}).T[['N_launches','Years_needed','E_R_local','E_H_local','E_O3','E_CO2']])

print('\n最优α（总影响最小）:')
print(f"  alpha = {best['alpha']:.2f}")
print(f"  年度火箭发射 N = {int(best['N_launches'])}")
print(f"  消耗时间 years = {int(best['Years_needed'])}")
print(f"  E_total = {best['E_total']:.4f}")

# 展示最优点的分项
print('\n最优点分项（raw / norm）:')
show_cols = ['E_R_local','E_H_local','E_O3','E_CO2','E_total']
print(df.loc[best_idx, show_cols])

# 查看几个关键α值
alphas_to_check = [0.0, 0.25, 0.5, 0.75, 1.0]
print('\n多个关键点的分项数据:')
for alpha in alphas_to_check:
    # 找到最接近的α值
    idx = (df['alpha'] - alpha).abs().idxmin()
    print(f"\nα = {df.loc[idx, 'alpha']:.3f} (最优点: {idx==best_idx})")
    print(df.loc[idx, show_cols])

In [ ]:
# ==================== 4) 可视化（对标Q2：α-目标曲线 + 分项曲线）====================

fig, ax = plt.subplots(1, 1)
ax.plot(df['alpha'], df['E_total'], linewidth=2)
ax.set_xlabel('alpha (Elevator Share)')
ax.set_yscale('log')
ax.set_ylabel('E_total (sum)')
ax.set_title('Q4: Environmental Impact vs Allocation α')
ax.axvline(best['alpha'], color='red', linestyle='--', linewidth=1)
ax.text(best['alpha'], df['E_total'].min(), f"best α={best['alpha']:.2f}",
        ha='left', va='bottom', color='red')
plt.show()

fig, ax = plt.subplots(1, 1)
ax.plot(df['alpha'], df['E_R_local'], label='E_R_local ')
ax.plot(df['alpha'], df['E_H_local'], label='E_H_local ')
ax.plot(df['alpha'], df['E_O3'], label='E_O3 ')
ax.plot(df['alpha'], df['E_CO2'], label='E_CO2 ')
ax.set_xlabel('alpha (Elevator Share)')
ax.set_yscale('log')
ax.set_ylabel('component value')
ax.set_title('Q4: Component Impacts ')
ax.legend()
plt.show()

# 输出前10条（便于检查）
print('\n搜索结果预览（前10行）:')
print(df[['alpha','N_launches','Years_needed','E_total']].head(21))
